In [ ]:
%load_ext autoreload
%autoreload 2

# Bolshoi-Rep variant models: z=0 comparison

Compares the five `data/bolshoi_rep/` abundance-matching samples --
`fid.h5` (fiducial) plus the `DF_down`/`DF_up` and `alpha_down`/`alpha_up`
variants, produced by `src/bolshoi_rep/DF/run_*.py` -- via their z0-summary
CSVs (`fid_z0.csv`, `DF_down_z0.csv`, `DF_up_z0.csv`, `alpha_down_z0.csv`,
`alpha_up_z0.csv`), built in `data/bolshoi_rep/` with
`jsm_processh5.ProcessH5` (see `notebooks/paper3/concentration_comparison_fid.ipynb`
for that same pattern applied to `fid.h5`).

`logc` in all five CSVs is `ProcessH5(..., conctype=None)` -- the
*analytic* Zhao et al. (2009) host concentration (`host_c` in the raw h5,
matching `conctype='zhao'` set in `SatGen/src/jsm_SubGen_bolshoi_rep.py`
when these trees were generated), not `conctype="measured"` (fit to each
tree's realized density profile) or `conctype="ludlow"` (the Ludlow et al.
2016 relation). Because it's the analytic, mass-only value, `logc` is
identical across all 5 variants for the same `tree_index` -- verified
2026-08-28 (the CSVs previously shipped with `conctype="measured"` by
default and were regenerated).

Unlike the epsilon-vs-fiducial comparison (single 13.0 mass bin), these
samples span `logMvir` ~ 13.1-13.3, so `logMvir` is kept as a corner-plot
key here.

Same `plot_corner_with_corr` / matched-by-`tree_index` approach as
`notebooks/paper3/epsilon_vs_fiducial.ipynb`, generalized to five models
instead of two. `notebooks/paper3/satgen_bolshoi.ipynb` has an earlier,
never-run sketch of a `DF_down`/`fid`/`DF_up` comparison that labels them
by dynamical-friction strength ($\beta$ = 0.075 / 0.75 / 7.5) -- carried
through below as a labeling hint, not independently verified here. No
equivalent value was found on disk for the `alpha_down`/`alpha_up` pair,
so those keep plain filename-based labels.

In [ ]:
import itertools
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '/Users/jsmonzon/Research/SatGen/mcmc/src/')
import jsm_ancillary
import jsm_stats

In [ ]:
plt.style.use('../../../SatGen/notebooks/paper1/paper.mplstyle')
double_textwidth = 7.0  # inches
single_textwidth = 3.5  # inches
levelz = [1 - 0.99, 1 - 0.95, 1 - 0.68]  # contour levels, matches satgen_bolshoi.ipynb

## Load the five z0-summary tables

In [ ]:
model_files = {
    "fid": "fid_z0.csv",
    "DF_down": "DF_down_z0.csv",
    "DF_up": "DF_up_z0.csv",
    "alpha_down": "alpha_down_z0.csv",
    "alpha_up": "alpha_up_z0.csv",
}

model_dfs = {name: pd.read_csv(f"../../data/bolshoi_rep/{fname}") for name, fname in model_files.items()}

for name, df in model_dfs.items():
    print(name, df.shape)

## Corner plot: fiducial vs DF/alpha variants

Same panel layout as `epsilon_vs_fiducial.ipynb`'s `plot_corner_with_corr`
(diagonal = 1D KDE marginals, lower triangle = 2D KDE contours + Spearman
$\rho$ annotated per model), generalized to five models. `logMvir` is kept
in the panel keys here since these samples span a mass range rather than a
single bin.

All five models are paired by `tree_index` (an inner merge across all
five) rather than by mass bin -- they're the same underlying Bolshoi-Rep
trees, just evolved under different orbit-physics variants, so pairing on
`tree_index` compares each tree to itself across models (and drops any
tree that errored out in one model but not another).

In [ ]:
def plot_corner_with_corr(matched, models, keys, paramNames, colors,
                           paramRanges=None, figsize=14):
    """
    Corner comparison between an arbitrary number of models using a matched
    dataframe, with Spearman correlation coefficients annotated in each
    off-diagonal panel.

    models : list of (suffix, label) tuples, e.g. [("fid", "fiducial"), ("DF_up", "DF up")]
    keys   : column base-names to compare, expected in matched as f"{key}_{suffix}"
    paramRanges : list of [lo, hi] per key, or None to auto-range from the data (1st/99th percentile + 5% pad)
    """

    model_colors = dict(zip([m for _, m in models], colors))

    model_chains = {}
    model_rho = {}

    for suffix, model in models:
        cols = [f"{k}_{suffix}" for k in keys]
        chain = matched[cols].copy()
        chain.columns = keys

        mask = ~chain.isna().any(axis=1)
        chain = chain.loc[mask].reset_index(drop=True)

        pair_rho = {}
        for k1, k2 in itertools.combinations(keys, 2):
            pair_rho[(k1, k2)] = jsm_stats.correlation(chain[k1].values, chain[k2].values)

        model_chains[model] = chain
        model_rho[model] = pair_rho

    if paramRanges is None:
        paramRanges = []
        for k in keys:
            allvals = np.concatenate([model_chains[m][k].values for _, m in models])
            allvals = allvals[np.isfinite(allvals)]
            lo, hi = np.nanpercentile(allvals, [1, 99])
            pad = 0.5 * (hi - lo if hi > lo else 1.0)
            paramRanges.append([lo - pad, hi + pad])

    nDim = len(keys)
    fig, axes = plt.subplots(nDim, nDim, figsize=(figsize, figsize))

    for i in range(nDim):
        for j in range(nDim):
            ax = axes[i, j]

            if j > i:
                ax.axis("off")
                continue

            if i == j:
                for _, model in models:
                    chain = model_chains[model]
                    sns.kdeplot(chain[keys[i]], ax=ax, color=model_colors[model], fill=False, lw=1.8)
                ax.set_xlim(paramRanges[i])
                ax.set_yticks([])

            else:
                for _, model in models:
                    chain = model_chains[model]
                    sns.kdeplot(x=chain[keys[j]], y=chain[keys[i]], ax=ax, color=model_colors[model],
                                levels=levelz, fill=False, linewidths=1.3)
                ax.set_xlim(paramRanges[j])
                ax.set_ylim(paramRanges[i])

                pair = (keys[j], keys[i])
                first_model = models[0][1]
                if pair not in model_rho[first_model]:
                    pair = (keys[i], keys[j])

                y0 = 0.92
                for _, model in models:
                    rho = model_rho[model][pair]
                    ax.text(0.6, y0, "$\\rho$=" + f"{rho:.2f}", transform=ax.transAxes, fontsize=8,
                            color=model_colors[model],
                            bbox=dict(boxstyle="round", facecolor="white", alpha=1, edgecolor="k", pad=0.15))
                    y0 -= 0.13

            if i < nDim - 1:
                ax.set_xticklabels([])
                ax.set_xlabel("")
            else:
                ax.set_xlabel(paramNames[j], fontsize=12, fontfamily="Times")

            if j > 0 or i == j:
                ax.set_yticklabels([])
                ax.set_ylabel("")
            else:
                ax.set_ylabel(paramNames[i], fontsize=12, fontfamily="Times")

            ax.tick_params(labelsize=10)
            for label in ax.get_xticklabels() + ax.get_yticklabels():
                label.set_fontfamily("Times")

    handles = [plt.Line2D([0], [0], color=model_colors[model], lw=1.8, label=model) for _, model in models]
    fig.legend(handles=handles, loc="upper right", fontsize=12, frameon=False)

    plt.subplots_adjust(wspace=0.08, hspace=0.08)
    plt.tight_layout()
    return fig

In [ ]:
def add_suffix(df, suf):
    return df.rename(columns={c: f"{c}_{suf}" for c in df.columns if c != "tree_index"})

models = [("fid", "fiducial"), ("DF_down", "DF down"), ("DF_up", "DF up"),
          ("alpha_down", "alpha down"), ("alpha_up", "alpha up")]

frames = [add_suffix(model_dfs[suf], suf) for suf, _ in models]
matched = frames[0]
for frame in frames[1:]:
    matched = matched.merge(frame, on="tree_index", how="inner")

print(f"{len(matched)} trees matched by tree_index across all 5 models "
      "(of " + ", ".join(f"{len(model_dfs[suf])} {suf}" for suf, _ in models) + ")")

In [ ]:
keys = ["logMvir", "log1pz50", "logc", "logNsub", "logfsub"]
paramNames = ["logM$_{\\rm vir}$", "log (1 + z$_{50}$)", "logc", "logNsub", "logfsub"]
colors = ("black", "steelblue", "royalblue", "darkorange", "orangered")

fig = plot_corner_with_corr(matched, models=models, keys=keys, paramNames=paramNames,
                             colors=colors, figsize=9.0)
# fig.savefig("../../figures/bolshoi_rep_corner.pdf", bbox_inches="tight")  # not auto-saving to disk -- left inline only

## fsub: fiducial vs DF/alpha variants

In [ ]:
for suf, label in models:
    sns.kdeplot(matched[f"fsub_{suf}"], label=label, color=dict(zip([m for _, m in models], colors))[label])
plt.xlabel("fsub")
plt.legend()